In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import TYPE_CHECKING, cast

import deepmolecules
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from parameteriser import (
    select_substrate,
)
from parameteriser.brenda.v0 import Brenda

if TYPE_CHECKING:
    from matplotlib.axes import Axes
    from matplotlib.figure import Figure

In [ ]:
brenda = Brenda()

if (path := Path.home() / "Documents" / "brenda_2023_1.json").exists():
    brenda.read_database(path)

## Check distribution of predicted kms for a wide range of organisms

- Viridiplantae (plants & algae)
  - embryophyta (land plants, taxonomy_id: 3193)
- get protein sequence for all plants / algea?

In [ ]:
def download_top_n_sequences(ec: str, taxonomy_id: int, n: int = 100) -> pd.DataFrame:
    import zlib
    from io import BytesIO
    from urllib.parse import quote

    import requests

    url = (
        "https://rest.uniprot.org/uniprotkb/search?"
        "compressed=true"
        + quote("&fields=accession,organism_name,sequence", safe="&=+")
        + "&format=tsv"
        + quote(f"&query=((ec:{ec})+AND+(taxonomy_id:{taxonomy_id}))", safe="&=+")
        + f"&size={n}"
    )

    return pd.read_csv(
        BytesIO(
            zlib.decompress(
                requests.get(url, timeout=60).content,
                wbits=16 + zlib.MAX_WBITS,
            ),
        ),
        sep="\t",
    )


def normalise(x: np.ndarray) -> np.ndarray:
    return x / np.sum(x)


def plot_data_and_predicted(
    brenda_kms: pd.Series,
    predicted_kms: pd.Series,
    title: str,
    max_kms_to_plot: int = 20,
) -> tuple[Figure, Axes]:
    from scipy.stats import gaussian_kde

    xmin = min(predicted_kms.min(), brenda_kms.min()) * 0.8
    xmax = max(predicted_kms.max(), brenda_kms.max()) * 1.2

    x = np.geomspace(xmin, xmax, 1001)
    y1 = normalise(gaussian_kde(brenda_kms)(x))
    y2 = normalise(gaussian_kde(predicted_kms)(x))

    with plt.rc_context(
        {
            "grid.color": "0.8",
            "xtick.color": "0.8",
            "ytick.color": "0.8",
            "xtick.labelcolor": "0.3",
            "ytick.labelcolor": "0.3",
        },
    ):
        fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
        ax.set_xlim(xmin, xmax)
        ax.set_xscale("log")

        ax.fill_between(x, y1, alpha=0.2)
        ax.fill_between(x, y2, alpha=0.2)
        ax.plot(x, y1, label=f"Brenda, n={len(brenda_kms)}")
        ax.plot(x, y2, label=f"deepmolecules, n={len(predicted_kms)}")
        ax.grid(visible=True)
        ax.set_frame_on(False)
        ax.legend()

    ax.set_title(title)

    if len(brenda_kms) < max_kms_to_plot:
        for val in brenda_kms.to_numpy():
            ax.axvline(val, ymin=0.045, ymax=0.065, color="black")
    return fig, ax


embryophyta = 3193  # embroyphyta

## rubisco carboxylation

In [ ]:
ec = "4.1.1.39"
brenda_substrate = "CO2"
kegg_substrate = "C00011"  # CO2

seqs = download_top_n_sequences(ec=ec, taxonomy_id=embryophyta, n=100)
seqs = seqs.iloc[seqs["Organism"].drop_duplicates().index]
brenda_kms = cast(
    pd.Series,
    (
        select_substrate(
            brenda.get_kms_and_kcats(ec=ec)[0],
            brenda_substrate,
        )["value"]
    ),
)
predicted_kms = deepmolecules.km.predict(
    [kegg_substrate] * len(seqs),
    seqs["Sequence"].values,
)["KM [mM]"]
fig, ax = plot_data_and_predicted(
    brenda_kms,
    predicted_kms,
    title=f"{ec} - {brenda_substrate}",
)
plt.show()

### pgk

In [ ]:
ec = "2.7.2.3"
brenda_substrate = "3-phospho-D-glycerate"
kegg_substrate = "C00197"


seqs = download_top_n_sequences(ec=ec, taxonomy_id=embryophyta)
seqs = seqs.iloc[seqs["Organism"].drop_duplicates().index]
brenda_kms = cast(
    pd.Series,
    (
        select_substrate(
            brenda.get_kms_and_kcats(ec=ec)[0],
            brenda_substrate,
        )["value"]
    ),
)
predicted_kms = deepmolecules.km.predict(
    [kegg_substrate] * len(seqs),
    seqs["Sequence"].values,
)["KM [mM]"]
fig, ax = plot_data_and_predicted(
    brenda_kms,
    predicted_kms,
    title=f"{ec} - {brenda_substrate}",
)
plt.show()

## MEP pathway intermediate

In [ ]:
ec = "4.6.1.12"
brenda_substrate = "2-phospho-4-(cytidine 5'-diphospho)-2-C-methyl-D-erythritol"
kegg_substrate = "C11436"


seqs = download_top_n_sequences(ec=ec, taxonomy_id=embryophyta, n=100)
seqs = seqs.iloc[seqs["Organism"].drop_duplicates().index]
brenda_kms = cast(
    pd.Series,
    (
        select_substrate(
            brenda.get_kms_and_kcats(ec=ec)[0],
            brenda_substrate,
        )["value"]
    ),
)
predicted_kms = deepmolecules.km.predict(
    [kegg_substrate] * len(seqs),
    seqs["Sequence"].values,
)["KM [mM]"]
fig, ax = plot_data_and_predicted(
    brenda_kms,
    predicted_kms,
    title=f"{ec} - {brenda_substrate}",
)
plt.show()

## Most characterised enzymes

In [ ]:
ecs = sorted({i.stem for i in brenda._km_dir.glob("*.json")} ^ {"spontaneous"})  # noqa: SLF001
print(len(ecs))

data_per_km = {}
for i in ecs:
    print(
        i,
        max_count := brenda.get_kms_and_kcats(
            i,
            filter_mutant=False,
            filter_missing_sequences=False,
        )[0]["substrate"]
        .value_counts()
        .max(),
    )
    data_per_km[i] = max_count

pd.Series(data_per_km).dropna().sort_values(ascending=False).head()
pd.Series(data_per_km).dropna().sort_values(ascending=False).head().index

In [ ]:
from dataclasses import dataclass


@dataclass
class EcScan:
    ec: str
    brenda_substrate: str
    kegg_substrate: str


for scan in [
    EcScan(
        ec="2.5.1.18",
        brenda_substrate="1-chloro-2,4-dinitrobenzene",
        kegg_substrate="C14397",
    ),
    EcScan(ec="1.5.1.3", brenda_substrate="7,8-dihydrofolate", kegg_substrate="C00415"),
    EcScan(
        ec="2.7.7.27",
        brenda_substrate="alpha-D-glucose 1-phosphate",
        kegg_substrate="C00103",
    ),
    EcScan(ec="1.1.1.42", brenda_substrate="isocitrate", kegg_substrate="C00311"),
]:
    seqs = download_top_n_sequences(ec=scan.ec, taxonomy_id=embryophyta, n=100)
    seqs = seqs.iloc[seqs["Organism"].drop_duplicates().index]
    brenda_kms = cast(
        pd.Series,
        (
            select_substrate(
                brenda.get_kms_and_kcats(ec=ec)[0],
                scan.brenda_substrate,
            )["value"]
        ),
    )
    predicted_kms = deepmolecules.km.predict(
        [scan.kegg_substrate] * len(seqs),
        seqs["Sequence"].values,
    )["KM [mM]"]
    fig, ax = plot_data_and_predicted(
        brenda_kms,
        predicted_kms,
        title=f"{scan.ec} - {scan.brenda_substrate}",
    )
    plt.show()